# Capture-bias controls at the deployed budget: C2 (residualisation) and C3 (shape-matched pairs) with the 70-label probe

The paper reports C2 and C3 with a fully supervised probe (Table 7). This notebook repeats them with the **five published
70-label probes**, on the final split, so every control is shown at the budget that is deployed. CPU only (~5 min).
Output: `results_revision_v1/controls_70labels.json` — upload it.

In [ ]:
# CELL 0 — CONFIG (same paths as revision_v1_reproduce.ipynb)
DRIVE      = '/content/drive/MyDrive'
TEA_DRIVE  = f'{DRIVE}/Tea Leaf Dataset'
STUDY      = f'{DRIVE}/TeaLeaf_DINOv2_Study'
RES        = f'{STUDY}/results_revision_v1'
FEATS      = f'{RES}/features_dinov1_crop.npz'
SPLIT      = f'{RES}/_split.json'
EJSON      = f'{RES}/E_distillation_5rep.json'
OUT        = f'{RES}/controls_70labels.json'
DEPLOY     = f'{RES}/deploy'
LOCAL      = '/content/tea_leaf_local'
CACHE      = '/content/crop_cache'
SEED, N_SHOT, CROP_PAD, IMG = 42, 10, 0.06, 224
SHOTS  = (1, 2, 5, 10, 20, 50, 100)
ALPHAS = (0.0, 0.2, 0.4, 0.6)


In [ ]:
# CELL 1 — setup and local copy of the dataset
from google.colab import drive; drive.mount('/content/drive')
import os, io, re, json, time, glob, shutil, hashlib, random, warnings
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torchvision.models as tvm, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import ndimage
from scipy.stats import spearmanr
import cv2
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
os.makedirs(DEPLOY, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
IMG_EXT = ('.jpg', '.jpeg', '.png')

def _cp(t):
    s, d = t
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copyfile(s, d)
t0 = time.time()
tasks = [(os.path.join(r, f), os.path.join(LOCAL, os.path.relpath(r, TEA_DRIVE), f))
         for r, _, fs in os.walk(TEA_DRIVE) for f in fs if f.lower().endswith(IMG_EXT)]
with ThreadPoolExecutor(32) as ex: list(ex.map(_cp, tasks))
print('local copy: %d files in %.0fs' % (len(tasks), time.time() - t0))


In [ ]:
# CELL 2 — records in the original order: classes alphabetical, files sorted within class
classes = sorted(d for d in os.listdir(LOCAL) if os.path.isdir(os.path.join(LOCAL, d)))
HEALTHY = classes.index(next(c for c in classes if 'healthy' in c.lower()))
RECS = [dict(rel=f'{c}/{f}', path=os.path.join(LOCAL, c, f), y=k, d=int(k != HEALTHY))
        for k, c in enumerate(classes) for f in sorted(os.listdir(os.path.join(LOCAL, c)))
        if f.lower().endswith(IMG_EXT)]
print('classes:', classes, '| healthy index', HEALTHY)
print('records:', len(RECS))
assert len(classes) == 7 and len(RECS) == 13084, 'dataset differs from the published release'


In [ ]:
# CELL 3 — restore the group-aware split from _split.json, check labels against the cached features
SPL = json.load(open(SPLIT)); F = np.load(FEATS)
Xtr, Xte = F['Xtr'], F['Xte']
test_rel = ['/'.join(p.split('/')[-2:]) for p in SPL['test_paths']]
by_rel = {r['rel']: r for r in RECS}
missing = [p for p in test_rel if p not in by_rel]
assert not missing, f'{len(missing)} test files not found, e.g. {missing[:3]}'
TEST = [by_rel[p] for p in test_rel]                       # exact published test order
tset = set(test_rel)
TRAIN = [r for r in RECS if r['rel'] not in tset]          # record order (original convention)
ytr, dtr = np.array([r['y'] for r in TRAIN]), np.array([r['d'] for r in TRAIN])
yte, dte = np.array([r['y'] for r in TEST]), np.array([r['d'] for r in TEST])
print('train %d | test %d | test diseased %.4f' % (len(TRAIN), len(TEST), dte.mean()))
assert (len(TRAIN), len(TEST)) == (SPL['n_train'], SPL['n_test'])
assert (yte == F['yte']).all() and (dte == F['dte']).all(), 'test labels differ from cached features'
TRAIN_ORDER_OK = len(ytr) == len(F['ytr']) and (ytr == F['ytr']).all()
print('train label sequence matches cache:', TRAIN_ORDER_OK, '(order within class is checked in Cell 6)')


In [ ]:
# CELL 4 — segmentation and leaf crop
# Prefer the ORIGINAL modules from Drive (shape_confound.leaf_mask, feature_shape_control.crop_to_leaf);
# fall back to the paper's recipe: Otsu on HSV V|S, largest component, fill holes, bbox + 6% pad, 224x224.
import sys
for m in ('shape_confound.py', 'feature_shape_control.py'):
    hit = glob.glob(f'{STUDY}/**/{m}', recursive=True)
    if hit: shutil.copy(hit[0], f'/content/{m}')
sys.path.insert(0, '/content')
try:
    import shape_confound as _sc, feature_shape_control as _fsc
    _ORIG = True
except Exception as e:
    _ORIG = False; print('original modules not found (%s) -> fallback recipe' % e)

def _fb_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[..., 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (mv > 0) | (ms > 0); lab, n = ndimage.label(m)
    if n == 0: return np.ones(m.shape, bool)
    big = 1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1))))
    return ndimage.binary_fill_holes(lab == big)

def _fb_crop(rgb, m, pad=CROP_PAD, size=IMG):
    ys, xs = np.where(m); H, W = m.shape
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    px, py = pad * (x1 - x0), pad * (y1 - y0)
    x0, y0, x1, y1 = int(max(0, x0 - px)), int(max(0, y0 - py)), int(min(W, x1 + px)), int(min(H, y1 + py))
    return Image.fromarray(rgb[y0:y1, x0:x1]).resize((size, size), Image.BILINEAR)

def leaf_crop(path):
    rgb = np.array(Image.open(path).convert('RGB'))
    if _ORIG:
        try:
            try: m = _sc.leaf_mask(rgb)
            except Exception: m = _sc.leaf_mask(Image.fromarray(rgb))
            im = _fsc.crop_to_leaf(rgb, np.asarray(m).astype(bool))
            im = im if isinstance(im, Image.Image) else Image.fromarray(np.asarray(im).astype(np.uint8))
            return im.convert('RGB').resize((IMG, IMG)) if im.size != (IMG, IMG) else im.convert('RGB')
        except Exception:
            pass
    return _fb_crop(rgb, _fb_mask(rgb))
print('crop function:', 'ORIGINAL modules' if _ORIG else 'fallback recipe')


# --- array versions (same code path as leaf_crop, used for composited images) ---
def mask_arr(rgb):
    if _ORIG:
        try:
            try: return np.asarray(_sc.leaf_mask(rgb)).astype(bool)
            except Exception: return np.asarray(_sc.leaf_mask(Image.fromarray(rgb))).astype(bool)
        except Exception: pass
    return _fb_mask(rgb)

def leaf_crop_arr(rgb):
    if _ORIG:
        try:
            m = mask_arr(rgb)
            im = _fsc.crop_to_leaf(rgb, m)
            im = im if isinstance(im, Image.Image) else Image.fromarray(np.asarray(im).astype(np.uint8))
            return im.convert('RGB').resize((IMG, IMG)) if im.size != (IMG, IMG) else im.convert('RGB')
        except Exception:
            pass
    return _fb_crop(rgb, _fb_mask(rgb))


In [ ]:
# CELL 5 — leaf-geometry descriptors for every image (CPU, parallel)
def _fb_shape(m):
    m8 = m.astype(np.uint8); cnts, _ = cv2.findContours(m8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    c = max(cnts, key=cv2.contourArea); a = float(cv2.contourArea(c)); p = float(cv2.arcLength(c, True))
    if a < 1 or p < 1: return None
    x, y, w, h = cv2.boundingRect(c); ha = float(cv2.contourArea(cv2.convexHull(c))) + 1e-6
    if len(c) >= 5:
        (_, _), (MA, ma), _ = cv2.fitEllipse(c); ecc = float(np.sqrt(1 - (min(MA, ma) / (max(MA, ma) + 1e-6)) ** 2))
    else: ecc = 0.0
    return np.array([a / m.size, w / (h + 1e-6), a / ha, 4 * np.pi * a / (p ** 2 + 1e-6),
                     p / (np.sqrt(a) + 1e-6), ecc, float(m.sum()) / m.size], np.float32)

def shape_of(rec):
    try:
        img = Image.open(rec['path']).convert('RGB')
        if _ORIG:
            m = _sc.leaf_mask(img)                      # original: mask at 224x224, holes filled
            f = None if m is None else _sc.shape_features(m)
        else:
            m = _fb_mask(np.array(img.resize((IMG, IMG)))); f = _fb_shape(m)
        return None if f is None else np.asarray(f, np.float32)
    except Exception:
        return None

t0 = time.time()
with ProcessPoolExecutor(os.cpu_count()) as ex:
    S_tr = list(ex.map(shape_of, TRAIN, chunksize=32)); S_te = list(ex.map(shape_of, TEST, chunksize=32))
nfail = sum(s is None for s in S_tr + S_te)
dim = next(len(s) for s in S_tr if s is not None)
fill = lambda L: np.stack([s if s is not None else np.full(dim, np.nan, np.float32) for s in L])
S_tr, S_te = fill(S_tr), fill(S_te)
med = np.nanmedian(S_tr, 0)                                  # impute the rare failures with the train median
S_tr = np.where(np.isnan(S_tr), med, S_tr); S_te = np.where(np.isnan(S_te), med, S_te)
print('shape descriptors: %d dims, %d failures of %d, %.0fs' % (dim, nfail, len(TRAIN) + len(TEST), time.time() - t0))


In [ ]:
# CELL 7 — C2 and C3 with the published 70-label probes
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
def draw(rep, n=N_SHOT):
    r = np.random.default_rng(SEED * 1000 + rep)
    return np.concatenate([r.choice(np.where(ytr == c)[0], n, replace=False) for c in range(len(classes))])
def score(m, X): return 1 - m.predict_proba(X)[:, HEALTHY]
# C3: healthy/diseased TEST pairs matched on leaf area (+-0.03 of frame) and aspect ratio (+-0.20), original rule
def match_pairs(SH, Y, tol_area=0.03, tol_aspect=0.20):
    area, aspect = SH[:, 0], SH[:, 1]; used, hh, dd = set(), [], []
    for i in np.where(Y == 0)[0]:
        cand = [j for j in np.where(Y == 1)[0] if j not in used and abs(area[j] - area[i]) < tol_area and abs(aspect[j] - aspect[i]) < tol_aspect]
        if cand:
            j = min(cand, key=lambda j: abs(area[j] - area[i]) + abs(aspect[j] - aspect[i])); used.add(j); hh.append(i); dd.append(j)
    return np.array(hh), np.array(dd)
hh, dd = match_pairs(S_te, dte); pidx = np.r_[hh, dd]; yp = dte[pidx]
print('matched test pairs: %d (of %d healthy test leaves)' % (len(hh), (dte == 0).sum()))
# C2: features residualised against the seven shape descriptors (fit on all images, no labels), renormalised
Sall = StandardScaler().fit_transform(np.vstack([S_tr, S_te])); Sall = np.c_[Sall, np.ones(len(Sall))]
Xall = np.vstack([Xtr, Xte]); beta, *_ = np.linalg.lstsq(Sall, Xall, rcond=None)
R = Xall - Sall @ beta; R /= np.linalg.norm(R, axis=1, keepdims=True) + 1e-9; Rtr, Rte = R[:len(Xtr)], R[len(Xtr):]
res = {k: [] for k in ('feat_full', 'shape_full', 'feat_resid', 'feat_pairs', 'shape_pairs')}
for rep in range(5):
    i = draw(rep)
    mf = LogisticRegression(C=1.0, max_iter=2000).fit(Xtr[i], ytr[i])
    ms = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000)).fit(S_tr[i], ytr[i])
    mr = LogisticRegression(C=1.0, max_iter=2000).fit(Rtr[i], ytr[i])
    res['feat_full'].append(roc_auc_score(dte, score(mf, Xte))); res['shape_full'].append(roc_auc_score(dte, score(ms, S_te)))
    res['feat_resid'].append(roc_auc_score(dte, score(mr, Rte)))
    res['feat_pairs'].append(roc_auc_score(yp, score(mf, Xte[pidx]))); res['shape_pairs'].append(roc_auc_score(yp, score(ms, S_te[pidx])))
OUTJ = {k: dict(mean=float(np.mean(v)), sd=float(np.std(v, ddof=1)), per_rep=[float(x) for x in v]) for k, v in res.items()}
OUTJ['n_pairs'] = int(len(hh))
lab = dict(feat_full='features, all test leaves', shape_full='shape only, all test leaves', feat_resid='C2 features residualised on shape',
           feat_pairs='C3 features, matched pairs', shape_pairs='C3 shape only, matched pairs')
for k, v in OUTJ.items():
    if k != 'n_pairs': print('%-36s AUC %.3f ± %.3f' % (lab[k], v['mean'], v['sd']))
assert abs(OUTJ['feat_full']['mean'] - 0.9853) < 1e-3, 'probe does not reproduce the published 0.985: stop'
json.dump(OUTJ, open(OUT, 'w'), indent=1); print('saved', OUT, '-> upload this file')
